# Assignment 2 - Testing

this notebook demonstrates the required testing that is needed for assignment 2.

In [1]:
from data_representation import DataTable

## Test 1 - Load CSV

this test checks that the historical cancer data is able to load correctly and the values are converted to floats.

In [2]:
historical = DataTable("Data/cancer_historical1.csv")

print("Expected: The CSV file loads & the values are converted to floats.")
print("Actual: ", historical.get_row(0))
print("Feature1 type:", type(historical.get(0, "Feature1")))

Expected: The CSV file loads & the values are converted to floats.
Actual:  {'Feature1': 5.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 2.0, 'Feature6': 1.0, 'Feature7': 3.0, 'Feature8': 1.0, 'Feature9': 1.0, 'TumorType': 0.0}
Feature1 type: <class 'float'>


## Test 2 - DataTable methods

this test checks that the required DataTable methods correctly return the results.

In [3]:
print("Expected: All DataTable methods return the correct values.")

print("get:", historical.get(0, "Feature1"))
print("get_column:", historical.get_column("Feature1")[:5])
print("get_row:", historical.get_row(0))
print("get_column_names:", historical.get_column_names())
print("get_column_average:", historical.get_column_average("Feature1"))
print("get_column_min:", historical.get_column_min("Feature1"))
print("get_column_max:", historical.get_column_max("Feature1"))

Expected: All DataTable methods return the correct values.
get: 5.0
get_column: [5.0, 5.0, 3.0, 6.0, 4.0]
get_row: {'Feature1': 5.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 2.0, 'Feature6': 1.0, 'Feature7': 3.0, 'Feature8': 1.0, 'Feature9': 1.0, 'TumorType': 0.0}
get_column_names: ['Feature1', 'Feature2', 'Feature3', 'Feature4', 'Feature5', 'Feature6', 'Feature7', 'Feature8', 'Feature9', 'TumorType']
get_column_average: 4.555926544240401
get_column_min: 1.0
get_column_max: 10.0


## Test 3 - missing historical values
this tests checks that the missing values are replaced with the average of the valid values within the same column.

In [4]:

import csv
import tempfile
import os

with tempfile.TemporaryDirectory() as folder:
    file_path = os.path.join(folder, "test_historical.csv")

    with open(file_path, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["Feature1", "Feature2", "TumorType"])
        writer.writerow([2, 4, 0])
        writer.writerow(["?", 6, 1])
        writer.writerow([8, 8, 0])

    test_data = DataTable(file_path)

    print("Input: Feature1 values = [2, ?, 8]")
    print("Expected: Missing value replaced with 5.0")
    print("Actual:", test_data.get(1, "Feature1"))

Input: Feature1 values = [2, ?, 8]
Expected: Missing value replaced with 5.0
Actual: 5.0


## Test 4 - Missing new-case values
this test checks that the missing values in new cases are replaced by using historical column average

In [5]:
import shutil

with tempfile.TemporaryDirectory() as folder:
    historical_file = os.path.join(folder, "cancer_historical1.csv")
    new_file = os.path.join(folder, "cancer_new_cases1.csv")

    shutil.copyfile("Data/cancer_historical1.csv", historical_file)

    with open(new_file, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["Feature1", "Feature2"])
        writer.writerow(["?", 6])
        writer.writerow([10, 8])

    new_data = DataTable(new_file)

    expected = historical.get_column_average("Feature1")
    actual = new_data.get(0, "Feature1")

    print("Input: New case Feature1 = ? ")
    print("Expected:", expected)
    print("Actual:", actual)
    print("Test passed:", abs(expected - actual) < 0.000001)

Input: New case Feature1 = ? 
Expected: 4.555926544240401
Actual: 4.555926544240401
Test passed: True


## Test 5 - Multiple missing values

this test checks that multiple missing values are replaced correctly while also keeping the original number & order of rows.

In [6]:
with tempfile.TemporaryDirectory() as folder:
    file_path = os.path.join(folder, "test_missing_column.csv")

    with open(file_path, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["Feature1", "Feature2", "TumorType"])
        writer.writerow([2, "?", 0])
        writer.writerow(["?", 6, 1])
        writer.writerow([8, 10, 0])

    test_data = DataTable(file_path)

    print("Input: 3 rows with missing values.")
    print("Expected: 3 rows, with missing values replaced by column averages.")
    print("Actual rows:")
    for i in range(3):
        print(test_data.get_row(i))

    print("Row count preserved:", len(test_data.data) == 3)
    print("Values replaced correctly:",
          test_data.get(0, "Feature2") == 8.0 and
          test_data.get(1, "Feature1") == 5.0)

Input: 3 rows with missing values.
Expected: 3 rows, with missing values replaced by column averages.
Actual rows:
{'Feature1': 2.0, 'Feature2': 8.0, 'TumorType': 0.0}
{'Feature1': 5.0, 'Feature2': 6.0, 'TumorType': 1.0}
{'Feature1': 8.0, 'Feature2': 10.0, 'TumorType': 0.0}
Row count preserved: True
Values replaced correctly: True


## Test 6 - Majority-class prediction

this test checks that the predictor returns the most common tumor type within the historical dataset


In [7]:
from predictors import ModePredictor

mode_predictor = ModePredictor(historical)

labels = historical.get_column("TumorType")
expected = max(set(labels), key=labels.count)

actual = mode_predictor.predict(historical.get_row(0))

print("Input: Historical tumor types")
print("Expected majority class:", expected)
print("Actual prediction:", actual)
print("Test passed:", actual == expected)

Input: Historical tumor types
Expected majority class: 0.0
Actual prediction: 0.0
Test passed: True


## Test 7 - Nearest-neighbor prediction

This test checks that the nearest-neighbor predictor uses the nine features in order to find the closest historical patient and predict their tumor type

In [8]:
from predictors import NearestNeighborPredictor

nearest_predictor = NearestNeighborPredictor(historical)

test_patient = historical.get_row(0).copy()

expected = test_patient["TumorType"]
actual = nearest_predictor.predict(test_patient)

print("Input: First historical patient's features")
print("Expected prediction:", expected)
print("Actual prediction:", actual)
print("Test passed:", actual == expected)

Input: First historical patient's features
Expected prediction: 0.0
Actual prediction: 0.0
Test passed: True


## Test 8 - Out of range feature

this test checks if an OutOfSampleError is raised when a patients feature value is outside the historical range

In [9]:
from exceptions import OutOfSampleError

test_patient = historical.get_row(0).copy()
test_patient["Feature1"] = 100.0

print("Input: Feature1 = 100.0")
print("Expected: OutOfSampleError because Feature1 is outside the historical range")

try:
    nearest_predictor.predict(test_patient)
    print("Test passed: False")
except OutOfSampleError as error:
    print("Actual:", error)
    print("Test passed: True")

Input: Feature1 = 100.0
Expected: OutOfSampleError because Feature1 is outside the historical range
Actual: Feature1 with value 100.0 is outside historical range [1.0, 10.0]
Test passed: True


## Test 9 - Multiple out of range features

this test checks that the first out of range feature is seen when multiple features have invalid values

In [10]:
test_patient = historical.get_row(0).copy()

test_patient["Feature2"] = 100.0
test_patient["Feature1"] = 100.0

print("Input: Feature1 = 100.0, Feature2 = 100.0")
print("Expected: OutOfSampleError for Feature1 first")

try:
    nearest_predictor.predict(test_patient)
    print("Test passed: False")
except OutOfSampleError as error:
    print("Actual:", error)
    print("Test passed:", "Feature1" in str(error))

Input: Feature1 = 100.0, Feature2 = 100.0
Expected: OutOfSampleError for Feature1 first
Actual: Feature1 with value 100.0 is outside historical range [1.0, 10.0]
Test passed: True


## Test 10 - Accuracy calculation

this test checks the accuracy of both predictors by comparing their predictions to the actual tumor types for the 100 new cases

In [11]:
new_cases = DataTable("Data/cancer_new_cases1.csv")
true_labels = DataTable("Data/cancer_new_cases_labels1.csv")

mode_correct = 0
nearest_correct = 0

for i in range(100):
    patient = new_cases.get_row(i)
    actual_label = true_labels.get(i, "TumorType")

    if mode_predictor.predict(patient) == actual_label:
        mode_correct += 1

    if nearest_predictor.predict(patient) == actual_label:
        nearest_correct += 1

mode_accuracy = mode_correct / 100 * 100
nearest_accuracy = nearest_correct / 100 * 100

print("Input: 100 new cases and their actual labels")
print("Expected: Accuracy percentages for both predictors")
print("Mode accuracy:", f"{mode_accuracy:.2f}%")
print("Nearest-neighbor accuracy:", f"{nearest_accuracy:.2f}%")


Input: 100 new cases and their actual labels
Expected: Accuracy percentages for both predictors
Mode accuracy: 79.00%
Nearest-neighbor accuracy: 98.00%


## Additional test - Missing values and OutOfSampleError

this test checks that a missing value that is replaced by the historical average doesn't raise an OutOfSampleError

In [12]:
with tempfile.TemporaryDirectory() as folder:
    historical_file = os.path.join(folder, "cancer_historical1.csv")
    new_file = os.path.join(folder, "cancer_new_cases1.csv")

    shutil.copyfile("Data/cancer_historical1.csv", historical_file)

    with open(new_file, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([f"Feature{i}" for i in range(1, 10)])
        row = historical.get_row(0)
        writer.writerow(["?" if i == 1 else row[f"Feature{i}"]
                         for i in range(1, 10)])
        
    test_new_data = DataTable(new_file)

    print("Input: Patient with missing Feature1")
    print("Expected: Prediction without OutOfSampleError")

    try:
        prediction = nearest_predictor.predict(test_new_data.get_row(0))
        print("Actual prediction:", prediction)
        print("Test passed: True")
    except OutOfSampleError as error:
        print("Actual:", error)
        print("Test passed: False")

Input: Patient with missing Feature1
Expected: Prediction without OutOfSampleError
Actual prediction: 0.0
Test passed: True
